In [ ]:
!pip install google-adk requests python-dotenv

In [ ]:
import os
import requests
import json
from datetime import datetime
import time

In [ ]:
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "True"
os.environ["GOOGLE_CLOUD_PROJECT"] = "qwiklabs-gcp-01-171e30612222"
os.environ["GOOGLE_CLOUD_LOCATION"] = "us-central1"

from google.adk.agents import LlmAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

import asyncio
from google.api_core.exceptions import ResourceExhausted

In [ ]:

def get_weather(city: str) -> str:
   """Retrieves current weather conditions for a specified city.

   Uses the Open-Meteo geocoding and weather APIs to fetch
   real-time temperature, wind speed, and weather conditions.
   No API key is required.

   Args:
       city: The name of the city to get weather for (e.g., 'Chicago', 'New York').

   Returns:
       A string containing current weather conditions including
       temperature, wind speed, humidity, and general conditions.
   """
   try:
       # Geocode the city name to coordinates
       geocode_url = "https://geocoding-api.open-meteo.com/v1/search"
       geo_response = requests.get(geocode_url, params={
           "name": city,
           "count": 1,
           "language": "en",
           "format": "json"
       })
       geo_data = geo_response.json()

       if "results" not in geo_data or len(geo_data["results"]) == 0:
           return f"Error: Could not find location '{city}'."

       location = geo_data["results"][0]
       lat = location["latitude"]
       lon = location["longitude"]
       resolved_name = location.get("name", city)
       admin1 = location.get("admin1", "")
       country = location.get("country", "")

       # Fetch current weather
       weather_url = "https://api.open-meteo.com/v1/forecast"
       weather_response = requests.get(weather_url, params={
           "latitude": lat,
           "longitude": lon,
           "current": "temperature_2m,relative_humidity_2m,apparent_temperature,precipitation,wind_speed_10m,wind_gusts_10m,weather_code",
           "temperature_unit": "fahrenheit",
           "wind_speed_unit": "mph",
           "precipitation_unit": "inch",
           "timezone": "auto"
       })
       weather_data = weather_response.json()

       if "current" not in weather_data:
           return f"Error: Could not retrieve weather data for '{city}'."

       current = weather_data["current"]

       weather_codes = {
           0: "Clear sky", 1: "Mainly clear", 2: "Partly cloudy",
           3: "Overcast", 45: "Foggy", 48: "Depositing rime fog",
           51: "Light drizzle", 53: "Moderate drizzle", 55: "Dense drizzle",
           61: "Slight rain", 63: "Moderate rain", 65: "Heavy rain",
           71: "Slight snow", 73: "Moderate snow", 75: "Heavy snow",
           80: "Slight rain showers", 81: "Moderate rain showers",
           82: "Violent rain showers", 95: "Thunderstorm",
           96: "Thunderstorm with slight hail", 99: "Thunderstorm with heavy hail"
       }

       code = current.get("weather_code", -1)
       condition = weather_codes.get(code, f"Unknown (code {code})")

       return (
           f"Current Weather for {resolved_name}, {admin1}, {country}:\n"
           f"  Condition: {condition}\n"
           f"  Temperature: {current['temperature_2m']}°F\n"
           f"  Feels Like: {current['apparent_temperature']}°F\n"
           f"  Humidity: {current['relative_humidity_2m']}%\n"
           f"  Wind: {current['wind_speed_10m']} mph (gusts {current['wind_gusts_10m']} mph)\n"
           f"  Precipitation: {current['precipitation']} inches\n"
           f"  Coordinates: ({lat}, {lon})"
       )

   except Exception as e:
       return f"Error fetching weather data: {str(e)}"

In [ ]:

def get_weather_alerts(state: str) -> str:
   """Retrieves active weather alerts for a US state from the National Weather Service.

   Queries the NWS API for all active alerts including severe weather
   warnings, watches, and advisories. No API key required.

   Args:
       state: The two-letter US state code (e.g., 'IL', 'CA', 'NY').

   Returns:
       A string listing all active weather alerts with severity,
       event type, and headline, or a message indicating no active alerts.
   """
   try:
       state = state.strip().upper()

       if len(state) != 2:
           return f"Error: '{state}' is not a valid two-letter US state code."

       url = "https://api.weather.gov/alerts/active"
       headers = {
           "User-Agent": "GCPBootcampWeatherAgent/1.0",
           "Accept": "application/geo+json"
       }
       response = requests.get(url, params={"area": state}, headers=headers)

       if response.status_code != 200:
           return f"Error: NWS API returned status {response.status_code}."

       data = response.json()
       features = data.get("features", [])

       if not features:
           return (
               f"No active weather alerts for {state} as of "
               f"{datetime.now().strftime('%Y-%m-%d %H:%M')}. All clear!"
           )

       output = [f"Active Weather Alerts for {state} ({len(features)} found):\n"]

       for i, feature in enumerate(features[:10], 1):
           props = feature.get("properties", {})
           output.append(
               f"--- Alert {i} ---\n"
               f"  Event: {props.get('event', 'Unknown')}\n"
               f"  Severity: {props.get('severity', 'Unknown')}\n"
               f"  Urgency: {props.get('urgency', 'Unknown')}\n"
               f"  Headline: {props.get('headline', 'N/A')}\n"
               f"  Areas: {props.get('areaDesc', 'Unknown')}\n"
               f"  Expires: {props.get('expires', 'N/A')}\n"
           )

       if len(features) > 10:
           output.append(f"\n... and {len(features) - 10} more alerts.")

       return "\n".join(output)

   except Exception as e:
       return f"Error fetching alerts: {str(e)}"

In [ ]:
weather_agent = LlmAgent(
   name="weather_alerts_agent",
   model="gemini-2.0-flash",
   instruction="""You are a real-time weather information specialist.
You provide accurate, current weather data and active severe weather alerts.

You have two tools:
1. get_weather — fetches current weather conditions for any city worldwide.
2. get_weather_alerts — fetches active weather alerts for a US state.
  Requires a two-letter state code (e.g., IL, CA, NY).

Guidelines:
- For weather questions about a city, use get_weather.
- For alert or severe weather questions about a US state, use get_weather_alerts.
- If the user asks about both, use both tools.
- Present data clearly and recommend precautions for severe alerts.
""",
   tools=[get_weather, get_weather_alerts],
)

In [ ]:
session_service = InMemorySessionService()

runner = Runner(
    agent=weather_agent,
    app_name="weather_alerts_app",
    session_service=session_service,
)

session = await session_service.create_session(
    app_name="weather_alerts_app",
    user_id="bootcamp_user",
)

In [ ]:
async def ask_agent(query: str):
    """Sends a query to the weather agent and prints the response with retry logic."""
    print(f"\n{'='*60}")
    print(f"USER: {query}")
    print(f"{'='*60}")

    content = types.Content(
        role="user",
        parts=[types.Part.from_text(text=query)]
    )

    final_response = ""
    max_retries = 5  # Define maximum number of retries
    delay_seconds = 3 # Initial delay in seconds

    for i in range(max_retries):
        try:
            async for event in runner.run_async(
                user_id="bootcamp_user",
                session_id=session.id,
                new_message=content,
            ):
                if event.is_final_response():
                    for part in event.content.parts:
                        if part.text:
                            final_response += part.text
            print(f"\nAGENT:\n{final_response}")
            print(f"{'='*60}")
            return # Exit the function if successful

        except ResourceExhausted as e:
            print(f"Attempt {i+1} failed due to ResourceExhausted: {e}")
            if i < max_retries - 1:
                print(f"Retrying in {delay_seconds} seconds...")
                await asyncio.sleep(delay_seconds) # Use asyncio.sleep for async functions
                delay_seconds *= 2 # Exponential backoff
            else:
                print("Max retries reached. Giving up.")
                raise # Re-raise the exception if all retries fail
        except Exception as e: # Catch other potential errors too
            print(f"Attempt {i+1} failed with an unexpected error: {e}")
            if i < max_retries - 1:
                print(f"Retrying in {delay_seconds} seconds...")
                await asyncio.sleep(delay_seconds)
                delay_seconds *= 2
            else:
                print("Max retries reached. Giving up.")
                raise


In [ ]:
await ask_agent("What is the current weather in Chicago, IL?")


USER: What is the current weather in Chicago, IL?



AGENT:
The current weather in Chicago, Illinois is foggy with a temperature of 41.1°F, but it feels like 37.7°F. The humidity is 99%, and the wind is 2.9 mph (gusts 3.6 mph). There is no precipitation.


In [ ]:
await ask_agent("Are there any active weather alerts for Illinois?")


USER: Are there any active weather alerts for Illinois?

AGENT:
Yes, there are several active weather alerts for Illinois. There are multiple flood warnings with severe severity and dense fog advisories. Please check the specific areas and expiration times to see if they affect you. For the flood warnings, avoid areas subject to flooding. For the dense fog advisories, use caution while driving, as visibility may be significantly reduced.



In [ ]:

await ask_agent(
   "I'm traveling to Portland, Oregon this weekend."
   "What's the weather and are there any severe weather alerts?"
)


USER: I'm traveling to Portland, Oregon this weekend.What's the weather and are there any severe weather alerts?
Attempt 1 failed with an unexpected error: 
On how to mitigate this issue, please refer to:

https://google.github.io/adk-docs/agents/models/#error-code-429-resource_exhausted


429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Resource exhausted. Please try again later. Please refer to https://cloud.google.com/vertex-ai/generative-ai/docs/error-code-429 for more details.', 'status': 'RESOURCE_EXHAUSTED'}}
Retrying in 3 seconds...

AGENT:
The current weather in Portland, Oregon is overcast with a temperature of 47.8°F, but it feels like 43.4°F. The humidity is 77%, and the wind is 5.4 mph (gusts 14.1 mph). There is no precipitation.

There are two active weather alerts for Oregon, but they do not appear to directly affect Portland. The alerts are a Winter Weather Advisory and a Wind Advisory, and they affect the mountain and foothill areas. If your travel plans in

In [ ]:
await ask_agent("What's the weather like in Detroit right now?")


USER: What's the weather like in Detroit right now?

AGENT:
The current weather in Detroit, Michigan is overcast with a temperature of 40.3°F, but it feels like 33.7°F. The humidity is 98%, and the wind is 9.7 mph (gusts 16.3 mph). There is no precipitation.
